# Create round_info.csv

Builds `round_info.csv` from the HAL configs (notebooks 01/05) and the FOV/boundary
layout (notebook 03) -- the per-round series/HAL-config/data-dir table that notebook
07 turns into the Dave recipe.

**Variable z per FOV**: the bits rows get `tissue_thickness="multi"` and
`z_lengths` (every tier's frame count, JSON-encoded ascending) -- read from
notebook 05's `metadata/z_tiers.csv`. `hal_config` for those rows is still set
to the **deepest** tier (the "full z-stack" upper bound), matching what
`create_dave_config`'s docstring expects: a real, representative file, just
not written into the movie template for a "multi" round (see `dave.py`'s
`_add_movie`) -- the positions file's own per-FOV column (notebook 05)
supplies the real per-FOV value instead.

In [ ]:
import os
import sys
import json
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.common.experiment_info import resolve_sample_identity
from MERci.acquisition.dave      import (
    create_round_info, create_round_info_multitissue, create_data_drive_skeleton,
)
from MERci.acquisition.positions import (
    discover_boundary_files, resolve_boundaries_source_dir, group_boundaries_by_path_mode,
)

In [ ]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
MULTI_Z_DIR   = SAMPLE_DIR / "multi_z"      # tier hal_configs/shutters (notebook 05)
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# SAMPLE_NAME is the TRUE top-level experiment id, auto-detected from the
# folder structure -- NOT SAMPLE_DIR.name once split into sibling
# acquisition-type subfolders. Must match what notebook 03 used, since
# create_round_info_multitissue below references positions_{SAMPLE_NAME}_*.txt.
SAMPLE_NAME, IMAGING_DIR = resolve_sample_identity(MERCI_DIR)
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

In [ ]:
# ── Experiment parameters ──────────────────────────────────────
MICROSCOPE  = "ST2"   # microscope identifier
DATA_DRIVES = []       # e.g. ["D:", "E:", "F:"] to round-robin hyb rounds across physical
                        # drives (cells/transit stay on SAMPLE_DIR's own drive); [] = single-drive

# Per-tissue path mode -- MUST match what notebook 03 actually used (same
# convention as BOUNDARY_SOURCE below).
TISSUE_PATH_MODE           = "legacy"
TISSUE_PATH_MODE_OVERRIDES = {}

def tissue_path_mode(t):
    mode = TISSUE_PATH_MODE_OVERRIDES.get(t, TISSUE_PATH_MODE)
    if mode not in ("legacy", "transit"):
        raise ValueError(f"Tissue {t}: path mode must be 'legacy' or 'transit', got {mode!r}")
    return mode

# ── Detect the tissue/boundary layout (written by notebook 03) ──────
BOUNDARY_SOURCE = None
BOUNDARY_DIR, BOUNDARY_SOURCE = resolve_boundaries_source_dir(POSITIONS_DIR, BOUNDARY_SOURCE)
print(f"BOUNDARY_SOURCE: {BOUNDARY_SOURCE}")
print(f"BOUNDARY_DIR   : {BOUNDARY_DIR}")

boundaries, MODE = discover_boundary_files(BOUNDARY_DIR)
groups           = group_boundaries_by_path_mode(boundaries, MODE, tissue_path_mode)
MULTI_BOUNDARY   = len(groups) > 1
print(f"Layout mode: {MODE}  ({len(boundaries)} boundary file(s) -> {len(groups)} segment(s)) -> "
      f"{'per-segment' if MULTI_BOUNDARY else 'single-positions'} recipe")

# Cells/transit HAL configs -- unaffected, still flat in settings/ (notebook 01).
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))
transit_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*transit*.xml"))
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"
TRANSIT_HAL_CONFIG = transit_hal_configs[0].name if transit_hal_configs else None

# Bits: read notebook 05's tier summary instead of globbing settings/ -- the
# DEEPEST tier (last row, sorted ascending by z_max_um) is the representative
# "full z-stack" hal_config kept in round_info's own hal_config column.
z_tiers_path = METADATA_DIR / "z_tiers.csv"
if not z_tiers_path.exists():
    raise FileNotFoundError(f"{z_tiers_path} not found -- run notebook 05 first.")
z_tiers_df = pd.read_csv(z_tiers_path).sort_values("z_max_um").reset_index(drop=True)

BITS_HAL_CONFIG = z_tiers_df.iloc[-1]["hal_stem"] + ".xml"   # deepest tier, e.g. "...deep-....xml"
Z_LENGTHS_JSON  = json.dumps(sorted(int(n) for n in z_tiers_df["n_frames"]))

print(f"\nCells   HAL config : {CELLS_HAL_CONFIG}")
print(f"Transit HAL config : {TRANSIT_HAL_CONFIG}")
print(f"Bits    HAL config (deepest tier, representative) : {BITS_HAL_CONFIG}")
print(f"z_lengths (all tiers' frame counts)                : {Z_LENGTHS_JSON}")

if MULTI_BOUNDARY and TRANSIT_HAL_CONFIG is None:
    raise FileNotFoundError(
        "Multiple boundaries detected but no hal-config-*transit*.xml in settings/. "
        "Run the transit cell in notebook 01 first."
    )

## Round – bit – color mapping

Define the round → bit → colour mapping for the codebook. This is the single
source of **`N_HYBS`** (the number of hybridisation/bits rounds, taken as the max
round index) used by the recipe below, and it is saved to `round_bit_color_map.csv`
for notebook 08 to reuse (data organization + Dave annotation).

In [ ]:
# round : hyb/bit index (1-indexed), matching the bits movie series number
#         (hal-{mic}_01, _02, …); NOT the Dave imaging-round number.
# bit   : bit number     |     color : excitation wavelength (nm)
round_bit_color = [
    (1,  1,  647), (1,  2,  560),
    (2,  3,  560), (2,  4,  647),
    (3,  5,  647), (3,  6,  560),
    (4,  7,  647), (4,  8,  560),
    (5,  9,  560), (5,  10, 647),
    (6,  11, 647), (6,  12, 560),
    (7,  13, 647), (7,  14, 560),
    (8,  15, 560), (8,  16, 647),
    (9,  17, 647), (9,  18, 560),
    (10, 19, 647), (10, 20, 560),
    (11, 21, 560), (11, 22, 647),
    (12, 23, 647), (12, 24, 560),
    (13, 25, 647), (13, 26, 560),
]

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

## Build round_info.csv, tagging bits rows as tissue_thickness="multi"

In [ ]:
if DATA_DRIVES:
    create_data_drive_skeleton(
        sample_dir  = SAMPLE_DIR,
        n_bits      = N_HYBS,
        data_drives = DATA_DRIVES,
        mode        = MODE,
        boundaries  = boundaries if MODE == "multi" else None,
    )

if MULTI_BOUNDARY:
    round_info = create_round_info_multitissue(
        microscope         = MICROSCOPE,
        n_bits              = N_HYBS,
        bits_hal_config     = BITS_HAL_CONFIG,
        cells_hal_config    = CELLS_HAL_CONFIG,
        transit_hal_config  = TRANSIT_HAL_CONFIG,
        sample_dir          = SAMPLE_DIR,
        boundaries          = boundaries,
        mode                = MODE,
        sample_name         = SAMPLE_NAME,
        data_drives         = DATA_DRIVES or None,
        tissue_path_mode    = tissue_path_mode,
    )
else:
    round_info = create_round_info(
        microscope       = MICROSCOPE,
        n_bits           = N_HYBS,
        bits_hal_config  = BITS_HAL_CONFIG,
        cells_hal_config = CELLS_HAL_CONFIG,
        sample_dir       = SAMPLE_DIR,
        data_drives      = DATA_DRIVES or None,
        positions_txt    = POSITIONS_DIR / f"positions_{SAMPLE_NAME}.txt",
    )

# Tag every bits row as a variable-z-per-FOV round (see dave.py's _add_movie /
# create_dave_config docstring) -- cells/transit rows are left untouched
# (blank tissue_thickness = "single", the normal case).
is_bits_row = round_info["imaging_type"] == "bits"
round_info["tissue_thickness"] = None
round_info.loc[is_bits_row, "tissue_thickness"] = "multi"
round_info["z_lengths"] = None
round_info.loc[is_bits_row, "z_lengths"] = Z_LENGTHS_JSON

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")